# 🏠 MietCheck · 🛠️ QUA³CK-Phase 4: Modellentwicklung

**Modul:** Data Analytics & Big Data (IU, 4. Semester) · **Projekt:** MietCheck – „Zahle ich zu viel Miete?"

> Teil 4 von 6 der QUA³CK-Notebook-Serie. Reihenfolge:
> **Q**ualitätsprüfung → **U**nderstanding → **A**lgorithmenauswahl →
> Modellentwicklung → **C**ross-Validation → **K**nowledge.

**Ziel:** das gewählte Modell (Gradient Boosting) sauber entwickeln – Datenaufbereitung, Encoding und Training als **eine** reproduzierbare Pipeline.

In [1]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import config as C
sns.set_theme(style='whitegrid', palette='rocket')
print('Setup ok ·', ROOT.name)

Setup ok · MietCheck


## 1. Die Aufbereitungs-Pipeline im Detail
- **Numerisch:** fehlende Werte → Median-Imputation, dann Standardisierung.
- **Kategorial:** fehlende Werte → häufigste Kategorie, dann One-Hot (seltene zusammengefasst).
- **Boolean:** unverändert durchgereicht.

In [2]:
from train import build_preprocessor
pre = build_preprocessor()
df = pd.read_parquet(C.CLEAN_PARQUET)
X, y = df[C.FEATURES], df[C.TARGET]
pre.fit(X)
n_feat = pre.transform(X[:5]).shape[1]
print(f'Aus {len(C.FEATURES)} Merkmalen werden nach One-Hot {n_feat} Modell-Features.')

Aus 14 Merkmalen werden nach One-Hot 460 Modell-Features.


## 2. Training & Lernkurve (Train vs. Test)
Zeigt, dass das Modell generalisiert (kein starkes Overfitting).

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from train import get_models
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.2,random_state=C.RANDOM_STATE)
gb = get_models(build_preprocessor())['Gradient Boosting']
gb.fit(Xtr,ytr)
for lbl,(Xs,ys) in {'Train':(Xtr,ytr),'Test':(Xte,yte)}.items():
    p = gb.predict(Xs)
    print(f'  {lbl:5} MAE {mean_absolute_error(ys,p):6.1f} € · R² {r2_score(ys,p):.3f}')

  Train MAE   87.1 € · R² 0.907


  Test  MAE   89.7 € · R² 0.898


## 3. Das ausgelieferte Modell
In `src/train.py` wird das finale Modell auf **allen** Daten trainiert und gespeichert. Wir laden es und testen eine Vorhersage:

In [4]:
from joblib import load
model = load(C.MODEL_FILE)
bsp = pd.DataFrame([{ 'livingSpace':70,'noRooms':2.0,'yearConstructed':1990,
  'regio1':'Berlin','regio2':'Berlin','condition':'well_kept',
  'interiorQual':'normal','typeOfFlat':'apartment','balcony':True,
  'hasKitchen':True,'cellar':False,'lift':False,'garden':False,'newlyConst':False }])
print(f'Vorhergesagte Kaltmiete (70 m², Berlin): {model.predict(bsp)[0]:.0f} €')

Vorhergesagte Kaltmiete (70 m², Berlin): 694 €


## ✅ Fazit Phase Modellentwicklung
- Die gesamte Aufbereitung steckt als wiederverwendbare **Pipeline** im Modell → keine Datenlecks, voll reproduzierbar.
- Train- und Test-Fehler liegen nah beieinander → gute Generalisierung.
- Modell liegt als `models/mietcheck_model.joblib` bereit.

➡️ Weiter mit **05 · Kreuzvalidierung**.